In [1]:
import re
from collections import defaultdict
from typing import List
from utils.exploit_gates2 import NetlistParser

In [2]:
Test_Design_Number = 34
target_file = f"./release_hidden_0923/release_hidden/design{Test_Design_Number}.v"

trojan_gates = []

parser = NetlistParser()
parser.parse_netlist(target_file)
chain = []

from utils.Tokenizer_functions import extract_trojan_gates

reference_Trojans_file = f"./release_hidden_0923/release_hidden/result{Test_Design_Number}.txt"
actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
print(f"Actual trojan gates: {sorted(actual_trojan_gates)}")

/home/nadertehrani/Navid/Research/Contest Folders/Contest5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Actual trojan gates: ['g1', 'g10', 'g100', 'g1000', 'g1001', 'g1002', 'g1003', 'g1004', 'g1009', 'g1011', 'g1012', 'g1015', 'g1016', 'g1020', 'g1022', 'g1023', 'g1024', 'g1025', 'g1026', 'g1027', 'g1028', 'g1029', 'g1030', 'g1031', 'g1032', 'g1033', 'g1034', 'g1035', 'g1036', 'g1037', 'g1040', 'g1041', 'g1042', 'g1043', 'g1044', 'g1045', 'g1046', 'g1047', 'g1048', 'g1049', 'g105', 'g1050', 'g1051', 'g1053', 'g1055', 'g1056', 'g1057', 'g1058', 'g106', 'g1060', 'g1061', 'g1062', 'g1065', 'g1066', 'g1067', 'g1068', 'g1070', 'g1073', 'g1075', 'g1076', 'g1077', 'g1080', 'g1081', 'g1083', 'g1085', 'g1086', 'g1087', 'g1088', 'g1089', 'g109', 'g1090', 'g1091', 'g1092', 'g1094', 'g1095', 'g1097', 'g1098', 'g1099', 'g11', 'g110', 'g1100', 'g1101', 'g1102', 'g1103', 'g1105', 'g1106', 'g1107', 'g1108', 'g1109', 'g111', 'g1110', 'g1111', 'g1112', 'g1113', 'g1114', 'g1116', 'g1119', 'g112', 'g1120', 'g1122', 'g1123', 'g1124', 'g1125', 'g1129', 'g113', 'g1130', 'g1132', 'g1134', 'g1135', 'g1136', 'g1

In [3]:
for gate_name in actual_trojan_gates:
    gate = parser.get_gate(gate_name)
    trojan_outputs = []
    trojan_inputs = []
    for output in gate.outputs:
        if output.name in actual_trojan_gates:
            trojan_outputs.append(output.name)
    for input in gate.inputs:
        if input in actual_trojan_gates:
            trojan_inputs.append(input)
    print(f"Gate: {gate.name}, type: {gate.gate_type}, Inputs: {trojan_inputs}, Outputs: {trojan_outputs}")


Gate: g5098, type: not, Inputs: ['g3980'], Outputs: ['g799', 'g1089', 'g1122', 'g3011', 'g3576', 'g5798']
Gate: g1110, type: buf, Inputs: ['g2765'], Outputs: ['g850', 'g4904']
Gate: g2408, type: buf, Inputs: ['g5513'], Outputs: ['g3756', 'g4138']
Gate: g1691, type: buf, Inputs: ['g5304'], Outputs: ['g1698', 'g2417']
Gate: g5811, type: buf, Inputs: ['g2842'], Outputs: ['g863', 'g2529']
Gate: g5700, type: xnor, Inputs: ['g4415', 'g1565'], Outputs: ['g805', 'g1325', 'g3623', 'g4048']
Gate: g4415, type: nand, Inputs: ['g5181', 'g600'], Outputs: ['g5700']
Gate: g4637, type: xnor, Inputs: ['g1022', 'g5675'], Outputs: ['g1033', 'g1760', 'g3024', 'g3101', 'g3109', 'g3134', 'g5970']
Gate: g5181, type: nand, Inputs: ['g1022', 'g3981'], Outputs: ['g4415']
Gate: g1022, type: nand, Inputs: ['g3038', 'g2059'], Outputs: ['g4637', 'g5181']
Gate: g6492, type: xnor, Inputs: ['g3840', 'g2886'], Outputs: ['g137', 'g694', 'g2361', 'g2725', 'g3917', 'g4802', 'g4994']
Gate: g3038, type: nand, Inputs: ['g3840

In [ ]:
def find_PIs(gate_name: str) -> set:
    """
    Find all primary inputs (PIs) in the fanin cone of the given gate using bfs.
    """
    gate = parser.get_gate(gate_name)
    pis = set()
    queue = [gate]
    visited = set()
    visited.add(gate.name)
    while queue:
        current_gate = queue.pop(0)
        inputs = current_gate.inputs
        if current_gate.gate_type == 'dff':
            inputs = [current_gate.inputs[3]]
        for input_net in inputs:
            driver_gate = parser.get_gate(input_net)
            if input_net in visited:
                continue
            if driver_gate is None:
                # This net has no driver, it must be a primary input
                # if input_net[0] != '1' and input_net != 'n1':
                if input_net[0] != '1':
                    pis.add(input_net)
            else:
                # Add the driver gate to the queue for further exploration
                queue.append(driver_gate)
                visited.add(driver_gate.name)
    return pis